# Pathfinding Algorithms — Interactive Workshop

**Goal:** Build intuition for A*, Dijkstra, BFS, Greedy, and DFS by watching them run on the same grid.

**Run time:** ~20 minutes

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import heapq

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 10

---

## 1. Grid Setup

We'll use a 20×14 grid with walls. S=start, E=end.

In [ ]:
# 0 = empty, 1 = wall, 2 = start, 3 = end
EMPTY, WALL, START, END = 0, 1, 2, 3

GRID_ROWS, GRID_COLS = 14, 20

def make_grid():
    g = np.zeros((GRID_ROWS, GRID_COLS), dtype=int)
    # Outer boundary
    g[0, :], g[-1, :], g[:, 0], g[:, -1] = 1, 1, 1, 1
    # Internal walls
    g[4:10, 5] = 1
    g[5, 5:12] = 1
    g[5:11, 12] = 1
    g[10, 12:18] = 1
    g[3:8, 16] = 1
    g[8, 16:19] = 1
    return g

def place_nodes(g, s_row, s_col, e_row, e_col):
    g[s_row, s_col] = START
    g[e_row, e_col] = END
    return (s_row, s_col), (e_row, e_col)

START_POS = (2, 2)
END_POS   = (2, 17)

grid = make_grid()
START_POS, END_POS = place_nodes(grid, *START_POS, *END_POS)

def draw_grid(g, visited=None, frontier=None, path=None, title='Grid'):
    fig, ax = plt.subplots(figsize=(14, 7))
    cmap = plt.cm.colors.ListedColormap(['#161b22', '#484f58', '#3fb950', '#f85149'])
    ax.imshow(g, cmap=cmap, vmin=0, vmax=3)
    
    if visited:
        for r, c in visited:
            if g[r, c] == EMPTY:
                ax.add_patch(plt.Circle((c, r), 0.35, color='#58a6ff', alpha=0.6))
    
    if frontier:
        for r, c in frontier:
            if g[r, c] == EMPTY:
                ax.add_patch(plt.Circle((c, r), 0.3, color='#d29922', alpha=0.9))
    
    if path:
        for r, c in path:
            if g[r, c] == EMPTY:
                ax.add_patch(plt.Rectangle((c-0.4, r-0.4), 0.8, 0.8, color='#7ee787', alpha=0.9))
    
    for r in range(g.shape[0]):
        for c in range(g.shape[1]):
            if g[r, c] == START:
                ax.text(c, r, 'S', ha='center', va='center', fontsize=10, fontweight='bold', color='#0d1117')
            elif g[r, c] == END:
                ax.text(c, r, 'E', ha='center', va='center', fontsize=10, fontweight='bold', color='#0d1117')
    
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

draw_grid(grid, title='The Maze — S top-left, E top-right')

---

## 2. Helper Functions

In [ ]:
def get_neighbors(r, c, grid):
    """4-directional neighbors (no diagonals)."""
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < grid.shape[0] and 0 <= nc < grid.shape[1]:
            if grid[nr, nc] != WALL:
                yield nr, nc

def manhattan(r1, c1, r2, c2):
    return abs(r1-r2) + abs(c1-c2)

def reconstruct_path(parent, node):
    path = []
    while node:
        path.append(node)
        node = parent[node]
    path.reverse()
    return path

---

## 3. BFS — Wavefront Explorer

BFS explores in rings. Every node at distance d is visited before any node at distance d+1.

In [ ]:
def bfs(grid, start, end, animate=False):
    sr, sc = start
    er, ec = end
    queue = deque([(sr, sc)])
    visited = {start}
    parent = {start: None}
    frontier = []
    all_visited = []
    
    while queue:
        node = queue.popleft()
        all_visited.append(node)
        
        if (node[0], node[1]) == (er, ec):
            break
        
        next_frontier = []
        for nr, nc in get_neighbors(*node, grid):
            if (nr, nc) not in visited:
                visited.add((nr, nc))
                parent[(nr, nc)] = node
                queue.append((nr, nc))
                next_frontier.append((nr, nc))
        
        if animate:
            draw_grid(grid, visited=visited, frontier=next_frontier,
                      title=f'BFS — {len(visited)} visited')
    
    path = reconstruct_path(parent, (er, ec)) if (er, ec) in parent else None
    return path, len(visited), all_visited

bfs_path, bfs_visited, _ = bfs(grid, START_POS, END_POS)
draw_grid(grid, visited=bfs_visited, path=bfs_path,
          title=f'BFS — {len(bfs_visited)} nodes visited, path length = {len(bfs_path)}')

---

## 4. Dijkstra — Cost-First Explorer

Dijkstra is BFS with a priority queue. On uniform-cost grids it behaves similarly to BFS, but you can see the difference when edge costs vary (not shown here).

In [ ]:
def dijkstra(grid, start, end, animate=False):
    sr, sc = start
    er, ec = end
    dist = {start: 0}
    parent = {start: None}
    visited = set()
    pq = [(0, start)]
    all_visited = []
    
    while pq:
        d, node = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        all_visited.append(node)
        
        if node == (er, ec):
            break
        
        for nr, nc in get_neighbors(*node, grid):
            if (nr, nc) in visited:
                continue
            new_dist = dist[node] + 1
            if new_dist < dist.get((nr, nc), float('inf')):
                dist[(nr, nc)] = new_dist
                parent[(nr, nc)] = node
                heapq.heappush(pq, (new_dist, (nr, nc)))
    
    path = reconstruct_path(parent, (er, ec)) if (er, ec) in parent else None
    return path, len(visited), all_visited

dijk_path, dijk_visited, _ = dijkstra(grid, START_POS, END_POS)
draw_grid(grid, visited=dijk_visited, path=dijk_path,
          title=f'Dijkstra — {len(dijk_visited)} nodes visited, path length = {len(dijk_path)}')

---

## 5. Greedy Best-First — The Optimist

Greedy ignores how far it's traveled — it only chases the heuristic. This can be catastrophically wrong when obstacles redirect it.

In [ ]:
def greedy(grid, start, end, animate=False):
    sr, sc = start
    er, ec = end
    parent = {start: None}
    visited = set()
    pq = [(manhattan(sr, sc, er, ec), start)]
    all_visited = []
    
    while pq:
        _, node = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        all_visited.append(node)
        
        if node == (er, ec):
            break
        
        for nr, nc in get_neighbors(*node, grid):
            if (nr, nc) not in visited:
                parent[(nr, nc)] = node
                heapq.heappush(pq, (manhattan(nr, nc, er, ec), (nr, nc)))
    
    path = reconstruct_path(parent, (er, ec)) if (er, ec) in parent else None
    return path, len(visited), all_visited

greedy_path, greedy_visited, _ = greedy(grid, START_POS, END_POS)
draw_grid(grid, visited=greedy_visited, path=greedy_path,
          title=f'Greedy — {len(greedy_visited)} nodes visited, path length = {len(greedy_path)}')

---

## 6. A* — The Best of Both Worlds

f(n) = g(n) + h(n). g = cost from start, h = estimated cost to goal.

In [ ]:
def astar(grid, start, end, animate=False):
    sr, sc = start
    er, ec = end
    g_cost = {start: 0}
    parent = {start: None}
    visited = set()
    f = manhattan(sr, sc, er, ec)
    pq = [(f, start)]
    all_visited = []
    
    while pq:
        _, node = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        all_visited.append(node)
        
        if node == (er, ec):
            break
        
        for nr, nc in get_neighbors(*node, grid):
            if (nr, nc) in visited:
                continue
            tentative_g = g_cost[node] + 1
            if tentative_g < g_cost.get((nr, nc), float('inf')):
                g_cost[(nr, nc)] = tentative_g
                parent[(nr, nc)] = node
                f_val = tentative_g + manhattan(nr, nc, er, ec)
                heapq.heappush(pq, (f_val, (nr, nc)))
    
    path = reconstruct_path(parent, (er, ec)) if (er, ec) in parent else None
    return path, len(visited), all_visited

astar_path, astar_visited, _ = astar(grid, START_POS, END_POS)
draw_grid(grid, visited=astar_visited, path=astar_path,
          title=f'A* — {len(astar_visited)} nodes visited, path length = {len(astar_path)}')

---

## 7. Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

def draw_on_ax(ax, g, visited, path, title):
    ax.imshow(g, cmap=plt.cm.colors.ListedColormap(['#161b22','#484f58','#3fb950','#f85149']), vmin=0, vmax=3)
    for r, c in visited:
        if g[r, c] == EMPTY:
            ax.add_patch(plt.Circle((c, r), 0.35, color='#58a6ff', alpha=0.55))
    for r, c in path:
        if g[r, c] == EMPTY:
            ax.add_patch(plt.Rectangle((c-0.4, r-0.4), 0.8, 0.8, color='#7ee787', alpha=0.85))
    ax.text(START_POS[1], START_POS[0], 'S', ha='center', va='center', fontsize=9, fontweight='bold', color='#0d1117')
    ax.text(END_POS[1], END_POS[0], 'E', ha='center', va='center', fontsize=9, fontweight='bold', color='#0d1117')
    ax.set_title(title, fontsize=12)
    ax.axis('off')

draw_on_ax(axes[0,0], grid, bfs_visited, bfs_path, f'BFS — {len(bfs_visited)} visited, path={len(bfs_path)}')
draw_on_ax(axes[0,1], grid, dijk_visited, dijk_path, f'Dijkstra — {len(dijk_visited)} visited, path={len(dijk_path)}')
draw_on_ax(axes[1,0], grid, greedy_visited, greedy_path, f'Greedy — {len(greedy_visited)} visited, path={len(greedy_path)}')
draw_on_ax(axes[1,1], grid, astar_visited, astar_path, f'A* — {len(astar_visited)} visited, path={len(astar_path)}')

plt.suptitle('Same Maze — All Four Algorithms', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 8. Challenge

**Exercise:** Create a new grid where Greedy takes a path significantly longer than A*. 
Hint: think about what happens when an obstacle forces Greedy to backtrack through its entire route.

In [ ]:
# Your grid here:
# challenge_grid = make_grid()
# Add walls that fool Greedy
# Try it!

---

## Key Takeaways

| Algorithm | Nodes Explored | Path Optimal? | Speed |
|---|---|---|---|
| BFS | Baseline | ✅ Yes | Slowest |
| Dijkstra | Baseline | ✅ Yes | Slow |
| Greedy | Low | ❌ No | Fastest |
| A* | Lowest | ✅ Yes | Fast |

**A* wins** because it combines what BFS knows (cost from start) with what Greedy knows (direction to goal).